# Tuần 1 — Thiết lập Google Colab & Smoke Test cho TV3 (Transformer & LLM)

Notebook này thực hiện các công việc Tuần 1 của **TV3** theo hướng dẫn tại `docs/guides/TV3-tuan1.md`, `docs/WEEK1.md` và quy tắc trong `AGENTS.md`:
1. **Mục 3.1:** Xác thực Git tạm thời an toàn (không lưu `GH_TOKEN` vào `.git/config`, URL hay log), kiểm tra và cài đặt `requirements.lock` (fallback sang `requirements.txt` nếu chưa tương thích với Python trên Colab), kiểm tra phần cứng GPU thực tế và ghi đầy đủ thông tin môi trường vào `results/smoke/env.json`.
2. **Mục 3.2:** Chạy thử (smoke test) `distilbert/distilbert-base-uncased` trên 1.000 mẫu `train` và 500 mẫu `validation` của **Twitter FNS** trong 1 epoch để đo thời gian (`epoch_seconds`) và bộ nhớ GPU đỉnh (`peak_gpu_mem_gb`), lưu kèm đầy đủ metadata tái lập vào `results/smoke/distilbert.json`.
3. **Mục 3.3:** Chạy thử `Qwen/Qwen2.5-7B-Instruct` lượng tử hóa 4-bit (`nf4`, `float16`) trên 20 tiêu đề `train` của **Twitter FNS**, kiểm tra hợp lệ các token ID `A`/`B`/`C`, trích xuất xác suất từ logits (gộp biến thể có và không có khoảng trắng đầu), lưu thông số tổng hợp và `top10` next-token (nếu `abc_mass < 0.5`, không kèm văn bản tiêu đề) vào `results/smoke/qwen.json`.

**Quy tắc bắt buộc (`AGENTS.md` & `TV3-tuan1.md`):**
- Không bao giờ hardcode, in ra màn hình, lưu vào `.git/config` hoặc ghi vào log bất kỳ secret nào (`HF_TOKEN`, `GH_TOKEN`).
- Chỉ sử dụng tập `train`/`validation` của **Twitter FNS**; tuyệt đối không truy cập tập `test` hoặc dữ liệu tiêu đề năm 2026.
- Không tự tạo hoặc sửa đổi các file split chính thức trong `data/splits/`.
- Đảm bảo xóa toàn bộ cell output (`Edit -> Clear all outputs`) trước khi commit nếu output có chứa văn bản tiêu đề.

## 3.1 Xác thực Git tạm thời, Clone Repository & Cài đặt Dependencies (Ô 1)

**Cơ chế bảo mật và xử lý phụ thuộc:**
1. **Xác thực Git không lưu dấu vết:** Thay vì ghép `GH_TOKEN` vào URL (sẽ bị lưu lại trong `.git/config` hoặc lộ trong `CalledProcessError`), ô bên dưới truyền header `AUTHORIZATION: basic ...` tạm thời thông qua biến môi trường `GIT_CONFIG_COUNT` / `GIT_CONFIG_KEY_0` / `GIT_CONFIG_VALUE_0` chỉ tồn tại trong tiến trình `git clone`. URL trong `.git/config` hoàn toàn sạch (`https://github.com/thienandinh29/financial-sentiment-dl.git`).
2. **Kiểm tra `requirements.lock`:** Kiểm tra sự tồn tại của `requirements.lock` trước khi cài. Nếu chưa tồn tại hoặc quá trình cài đặt thất bại (ví dụ do Colab chạy Python 3.12 trong khi `requirements.lock` khóa cho Python 3.11), hệ thống sẽ ghi nhận phiên bản Python và thông báo lỗi vào biến `DEP_STATUS` để lưu vào `results/smoke/env.json` (báo cho TV2 tại CP2), sau đó tự động fallback sang `requirements.txt`.

In [ ]:
import base64
import os
import platform
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

# 1. Load HF_TOKEN into environment without printing or exposing it
token_val = userdata.get("HF_TOKEN")
if not token_val:
    raise RuntimeError("Missing HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = token_val
del token_val

# 2. Clone private repository using ephemeral Git HTTP header in subprocess env
REPO_URL = "https://github.com/thienandinh29/financial-sentiment-dl.git"
REPO_DIR = Path("financial-sentiment-dl")

if not REPO_DIR.exists():
    gh_val = userdata.get("GH_TOKEN")
    if not gh_val:
        raise RuntimeError("Missing GH_TOKEN in Colab Secrets.")
    basic_auth = base64.b64encode(f"x-access-token:{gh_val}".encode("utf-8")).decode("ascii")
    git_env = os.environ.copy()
    git_env.update({
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {basic_auth}",
    })
    del gh_val, basic_auth
    res = subprocess.run(
        ["git", "clone", "-q", REPO_URL, str(REPO_DIR)],
        env=git_env,
        capture_output=True,
        text=True,
        check=False,
    )
    del git_env
    if res.returncode != 0:
        raise RuntimeError(f"git clone failed with exit code {res.returncode}.")

if Path.cwd().name != REPO_DIR.name and REPO_DIR.exists():
    os.chdir(REPO_DIR)

# Verify that .git/config remote URL contains no credentials
remote_Origin = subprocess.run(
    ["git", "config", "--get", "remote.origin.url"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
assert remote_Origin == REPO_URL, "Unexpected remote.origin.url format."

# 3. Check requirements.lock and fallback to requirements.txt if missing or incompatible
py_ver = platform.python_version()
print(f"Python version: {py_ver}")

DEP_STATUS = {
    "python_version": py_ver,
    "requirements_lock_exists": Path("requirements.lock").is_file(),
    "installed_from": None,
    "requirements_lock_error": None,
}

if DEP_STATUS["requirements_lock_exists"]:
    lock_proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.lock"],
        capture_output=True,
        text=True,
        check=False,
    )
    if lock_proc.returncode == 0:
        DEP_STATUS["installed_from"] = "requirements.lock"
        print("Installed dependencies from requirements.lock successfully.")
    else:
        err_lines = (lock_proc.stderr or lock_proc.stdout or "").strip().splitlines()
        DEP_STATUS["requirements_lock_error"] = "\n".join(err_lines[-15:])
        print(f"requirements.lock failed on Python {py_ver}; falling back to requirements.txt.")
else:
    DEP_STATUS["requirements_lock_error"] = "requirements.lock not found on current branch."
    print("requirements.lock not found; falling back to requirements.txt.")

if DEP_STATUS["installed_from"] is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )
    DEP_STATUS["installed_from"] = "requirements.txt"
    print("Installed dependencies from requirements.txt successfully.")

# 4. Prepare pinned raw FNS data (train.jsonl & validation.jsonl with stable IDs for ID-only splits)
if not (Path("data/raw/fns/train.jsonl").is_file() and Path("data/raw/fns/validation.jsonl").is_file()):
    subprocess.run([sys.executable, "data/download.py", "--only", "fns"], check=True)

## 3.1 Kiểm tra phần cứng & Ghi nhận môi trường vào `results/smoke/env.json` (Ô 2)

Ô này kiểm tra trạng thái CUDA (`torch.cuda.is_available()` và `torch.cuda.device_count()`) trước khi truy cập thuộc tính GPU, ghi nhận tên GPU thực tế (không mặc định giả sử là T4), phiên bản các thư viện, commit SHA hiện tại và trạng thái cài đặt `requirements.lock` vào `results/smoke/env.json`.

In [ ]:
from datetime import datetime, timezone
import json
import os
import platform
import subprocess
import datasets
import numpy as np
import sklearn
import torch
import transformers

cuda_available = bool(torch.cuda.is_available() and torch.cuda.device_count() > 0)
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
else:
    gpu_name = None
    gpu_mem_gb = None

commit_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

env = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "commit_sha": commit_sha,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cuda_available": cuda_available,
    "gpu": gpu_name,
    "gpu_mem_gb": gpu_mem_gb,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "dependency_install": DEP_STATUS if "DEP_STATUS" in globals() else None,
}

os.makedirs("results/smoke", exist_ok=True)
with open("results/smoke/env.json", "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

print(json.dumps(env, indent=2))
if gpu_name != "Tesla T4":
    print(f"WARNING: Expected 'Tesla T4' for T4 budget estimation, but detected GPU: {gpu_name!r}")

## 3.2 Chạy thử DistilBERT — 1.000 mẫu train, 500 mẫu validation, 1 epoch

**Mục đích & Quy tắc dữ liệu (`AGENTS.md`, `TV3-tuan1.md` & `TV3-tuan2.md` Mục 3.8):**
- **Đọc Split dạng ID-only & Kiểm tra SHA-256 (`train.data.load_split_data`):** Theo Hợp đồng chung ở `docs/WEEK2.md`, file `data/splits/cluster_split.jsonl` chỉ lưu `{"id", "split"}` (không lưu văn bản). Hàm `load_train_val_splits` gọi `train.data.load_split_data` để kiểm tra bắt buộc mã băm SHA-256 (`8c2c35ea789acf5b2bd9c74c5e59fa0df325034422be42ae7b47f2e8c9c46aaa`), ghép văn bản từ `data/raw/fns/{train,validation}.jsonl` và ánh xạ nhãn về đúng thứ tự chuẩn `LABELS = ("bearish", "neutral", "bullish")` (`0, 1, 2`). Tuyệt đối không nạp tập `test`.
- **Đo thời gian & VRAM thực tế:** Huấn luyện 1 epoch với `SEED=1`, `N_TRAIN=1000`, `N_VAL=500`, `MAX_LEN=64`, `BS=32`, `lr=5e-5` và FP16 AMP (nếu có CUDA).
- **Đầy đủ Metadata tái lập (`AGENTS.md:L33`):** Ghi `commit_sha`, `seed`, `config`, `package_versions`, `gpu`, `epoch_seconds`, `peak_gpu_mem_gb` và `val_macro_f1_smoke` vào `results/smoke/distilbert.json`.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import platform
import subprocess
import time
import datasets
from datasets import Dataset
import numpy as np
import sklearn
from sklearn.metrics import f1_score
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import yaml
from train.data import DEFAULT_RAW, LABELS, load_split_data

SEED, N_TRAIN, N_VAL, MAX_LEN, BS, LR, EPOCHS = 1, 1000, 500, 64, 32, 5e-5, 1
DATASET_NAME = "zeroshot/twitter-financial-news-sentiment"
MODEL_NAME = "distilbert/distilbert-base-uncased"
DEFAULT_SPLIT_FILE = "data/splits/cluster_split.jsonl"
DEFAULT_SPLIT_SHA256 = "8c2c35ea789acf5b2bd9c74c5e59fa0df325034422be42ae7b47f2e8c9c46aaa"
DEVICE = "cuda" if (torch.cuda.is_available() and torch.cuda.device_count() > 0) else "cpu"

torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)


def load_train_val_splits(
    seed: int,
    n_train: int,
    n_val: int,
    split_file: str = DEFAULT_SPLIT_FILE,
    split_sha256: str = DEFAULT_SPLIT_SHA256,
):
    """Smoke-test subsets from the frozen ID-only split (SHA-256 checked inside load_split_data)."""
    cfg_path = Path("configs/E2.yaml")
    if cfg_path.is_file():
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg_data = yaml.safe_load(f) or {}
        data_cfg = cfg_data.get("data", {})
        if data_cfg.get("split_file"):
            split_file = data_cfg["split_file"]
        if "split_sha256" in data_cfg:
            split_sha256 = data_cfg["split_sha256"]

    if not split_sha256:
        raise ValueError(
            f"Official split file '{split_file}' exists, but 'split_sha256' is missing in '{cfg_path}'. "
            "Refusing to proceed with an unverified split."
        )

    data = load_split_data(split_file, split_sha256, DEFAULT_RAW, splits=("train", "validation"))

    def to_ds(rows, n):
        ds = Dataset.from_list([{"id": i, "text": t, "label": y} for i, t, y in rows])
        return ds.shuffle(seed=seed).select(range(min(n, len(ds))))

    return to_ds(data["train"], n_train), to_ds(data["validation"], n_val), {
        "source": str(split_file),
        "sha256": split_sha256,
        "verified": True,
        "labels": list(LABELS),
    }


train, val, split_info = load_train_val_splits(SEED, N_TRAIN, N_VAL)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(DEVICE)


def batches(split, bs, shuffle):
    order = np.random.default_rng(SEED).permutation(len(split)) if shuffle else np.arange(len(split))
    for i in range(0, len(order), bs):
        b = split.select(order[i:i + bs].tolist())
        enc = tok(list(b["text"]), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(list(b["label"]), device=DEVICE)


opt = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
if use_amp:
    torch.cuda.reset_peak_memory_stats()

model.train()
t0 = time.time()
for x, y in batches(train, BS, shuffle=True):
    with torch.autocast(DEVICE, dtype=torch.float16, enabled=use_amp):
        loss = model(**x, labels=y).loss
    opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
if use_amp:
    torch.cuda.synchronize()
epoch_s = time.time() - t0

model.eval()
preds = []
with torch.no_grad():
    for x, _ in batches(val, 64, shuffle=False):
        preds += model(**x).logits.argmax(-1).tolist()

commit_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
gpu_name = torch.cuda.get_device_name(0) if use_amp else None

log = {
    "model": MODEL_NAME,
    "n_train": len(train),
    "n_val": len(val),
    "batch_size": BS,
    "max_len": MAX_LEN,
    "seed": SEED,
    "epoch_seconds": round(epoch_s, 1),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if use_amp else None,
    "val_macro_f1_smoke": round(f1_score(list(val["label"]), preds, average="macro"), 3),
    "commit_sha": commit_sha,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": gpu_name,
    "config": {
        "dataset": DATASET_NAME,
        "split_info": split_info,
        "lr": LR,
        "epochs": EPOCHS,
        "optimizer": "AdamW",
        "mixed_precision_fp16": use_amp,
    },
    "package_versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__,
    },
}
print(json.dumps(log, indent=2))
with open("results/smoke/distilbert.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

## 3.3 Chạy thử Qwen2.5-7B-Instruct 4-bit & Đọc Logits A/B/C

**Mục đích & Cơ sở kỹ thuật (`context/decisions.md` & `TV3-tuan1.md`):**
- Nạp `Qwen/Qwen2.5-7B-Instruct` với lượng tử hóa 4-bit NormalFloat (`nf4`) và `bnb_4bit_compute_dtype=torch.float16`.
- Kiểm tra tính hợp lệ của `LETTER_IDS` trước khi tính xác suất: mỗi chữ cái `A`, `B`, `C` phải có ít nhất một single-token ID và các tập ID không được giao nhau.
- Lấy `logits[0, -1]`, dùng `torch.logsumexp` gộp xác suất của biến thể có/không khoảng trắng (`"A"` và `" A"`), sau đó chuẩn hóa bằng `softmax` trên 3 lớp `bearish` / `neutral` / `bullish`.
- Chỉ chạy trên đúng **20 tiêu đề tập `train` của Twitter FNS** (`seed=1`). Tuyệt đối không truy cập tập `test` hoặc tiêu đề năm 2026.
- Nếu `min_abc_mass < 0.5` (ngưỡng trong `TV3-tuan1.md`), lưu danh sách `top10` next-token strings của các mẫu có `abc_mass < 0.5` vào `results/smoke/qwen.json` để phục vụ tinh chỉnh prompt ở Tuần 2. **Tuyệt đối không lưu văn bản tiêu đề (headline text) vào JSON hoặc in ra màn hình.**

In [ ]:
from datetime import datetime, timezone
import gc
import json
import platform
import subprocess
import time
import datasets
from datasets import load_dataset
import numpy as np
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Guard: Qwen2.5-7B 4-bit NF4 requires a CUDA GPU
if not (torch.cuda.is_available() and torch.cuda.device_count() > 0):
    raise RuntimeError(
        "CUDA GPU is required to load Qwen/Qwen2.5-7B-Instruct in 4-bit NF4 (bitsandbytes). "
        "No CUDA device was detected; please enable a GPU runtime in Google Colab."
    )
gpu_name = torch.cuda.get_device_name(0)

# Free any cached VRAM from the previous DistilBERT smoke cell if run in the same session
gc.collect()
torch.cuda.empty_cache()

SEED = 1
N_SAMPLES = 20
ABC_MASS_THRESHOLD = 0.5
DATASET_NAME = "zeroshot/twitter-financial-news-sentiment"
name = "Qwen/Qwen2.5-7B-Instruct"

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
tok = AutoTokenizer.from_pretrained(name)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
model.eval()

PROMPT = (
    "Classify the sentiment of this financial news headline for the main company or market it is about.\n"
    "Headline: {text}\n"
    "A) bearish\nB) neutral\nC) bullish\n"
    "Answer with one letter only."
)
LETTERS = {"A": "bearish", "B": "neutral", "C": "bullish"}

# Token ids for "A" and " A" (with/without leading space); keep only single-token variants.
LETTER_IDS = {
    L: sorted({ids[0] for v in (L, " " + L) if len(ids := tok.encode(v, add_special_tokens=False)) == 1})
    for L in LETTERS
}

# Validate token IDs before computing probabilities
for letter, ids in LETTER_IDS.items():
    if not ids:
        raise ValueError(f"No single-token IDs found for option {letter!r}.")
all_letter_ids = [tid for ids in LETTER_IDS.values() for tid in ids]
assert len(all_letter_ids) == len(set(all_letter_ids)), f"Overlapping token IDs across letters: {LETTER_IDS}"


def letter_probs(text: str):
    """Return (probs over bearish/neutral/bullish, mass on the A/B/C tokens, top-10 next tokens)."""
    chat = tok.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(text=text)}],
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tok(chat, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    logp = torch.log_softmax(logits, dim=-1)
    per_letter = torch.stack([torch.logsumexp(logp[LETTER_IDS[L]], dim=0) for L in LETTERS])
    probs = torch.softmax(per_letter, dim=0)                        # renormalized over A/B/C
    top10 = [tok.decode([i]) for i in logits.topk(10).indices.tolist()]
    return dict(zip(LETTERS.values(), probs.tolist())), per_letter.exp().sum().item(), top10


# Load strictly from Twitter FNS train split (never test, never 2026 headlines)
train_split = load_dataset(DATASET_NAME, split="train")
texts = list(train_split.shuffle(seed=SEED).select(range(N_SAMPLES))["text"])

t0, rows = time.time(), []
for idx, text in enumerate(texts):
    probs, mass, top10 = letter_probs(text)
    rows.append({
        "sample_index": idx,
        "probs": {k: round(v, 6) for k, v in probs.items()},
        "abc_mass": round(mass, 3),
        "top10": top10,
    })
if torch.cuda.is_available():
    torch.cuda.synchronize()
seconds_per_headline = (time.time() - t0) / len(texts)

assert all(abs(sum(r["probs"].values()) - 1.0) < 1e-4 for r in rows)
min_abc_mass = min(r["abc_mass"] for r in rows)

# If min_abc_mass is below threshold (0.5), record top-10 tokens (without headline text) for Week 2 prompt tuning
low_mass_Diagnostics = [
    {"sample_index": r["sample_index"], "abc_mass": r["abc_mass"], "top10": r["top10"]}
    for r in rows
    if r["abc_mass"] < ABC_MASS_THRESHOLD
]

commit_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
cuda_available = bool(torch.cuda.is_available() and torch.cuda.device_count() > 0)
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

qwen_summary = {
    "model": name,
    "quant": "nf4-4bit",
    "seconds_per_headline": round(seconds_per_headline, 2),
    "min_abc_mass": min_abc_mass,
    "abc_mass_threshold": ABC_MASS_THRESHOLD,
    "letter_ids": LETTER_IDS,
    "low_mass_top10_diagnostics": low_mass_Diagnostics if min_abc_mass < ABC_MASS_THRESHOLD else [],
    "commit_sha": commit_sha,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "gpu": gpu_name,
    "config": {
        "dataset": DATASET_NAME,
        "split": "train",
        "n_samples": N_SAMPLES,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
    },
    "package_versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "numpy": np.__version__,
    },
}

print("letter ids:", LETTER_IDS, "| s/headline:", round(seconds_per_headline, 2))
print("min mass on A/B/C:", min_abc_mass)
if min_abc_mass < ABC_MASS_THRESHOLD:
    print(f"WARNING: min_abc_mass ({min_abc_mass}) < {ABC_MASS_THRESHOLD}. Saved {len(low_mass_Diagnostics)} top-10 diagnostics for Week 2.")

with open("results/smoke/qwen.json", "w", encoding="utf-8") as f:
    json.dump(qwen_summary, f, indent=2)